<a href="https://colab.research.google.com/github/cybercolombia/suelosabio/blob/feature%2FSCRUM-16/notebooks/ClimatePipeline/07_Climate_Precipitation_MunicipalAggregator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Climate_Precipitation_MunicipalAggregator

Construye precipitación municipio-día para los 239 municipios de Boyacá y Cundinamarca a partir de la capa estación-día curada y la geografía canónica.

## Alcance del piloto

- Solo implementa reglas de precipitación.
- Solo contribuyen estaciones con `asignacion_canonica=True`.
- Genera el calendario completo, incluso para municipios sin estación.
- No imputa ausencias y no convierte `NaN` en cero.
- Conserva media, mediana, extremos, dispersión y cobertura.
- Usa la mediana no ponderada como valor principal cuando la cobertura de estaciones esperadas es al menos 50 %.

Este paso no construye todavía indicadores semestrales o anuales.

## 1. Preparar el repositorio

La celda actualiza explícitamente `feature/SCRUM-16`, incluso si el clon de Colab fue creado antes como copia superficial de otra rama.

In [ ]:
from pathlib import Path

import subprocess
import sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/cybercolombia/suelosabio.git'
REPO_REF = 'feature/SCRUM-16'
REPO_DIR = Path('/content/suelosabio') if IN_COLAB else Path.cwd()
ACTUALIZAR_REPOSITORIO = True

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    elif ACTUALIZAR_REPOSITORIO:
        remote_ref = f'refs/remotes/origin/{REPO_REF}'
        subprocess.run(
            [
                'git', 'fetch', '--depth', '1', 'origin',
                f'+refs/heads/{REPO_REF}:{remote_ref}',
            ],
            cwd=REPO_DIR,
            check=True,
        )
        subprocess.run(
            ['git', 'checkout', '-B', REPO_REF, remote_ref],
            cwd=REPO_DIR,
            check=True,
        )

PIPELINE_DIR = REPO_DIR / 'notebooks' / 'ClimatePipeline'
if not PIPELINE_DIR.exists():
    raise FileNotFoundError(f'No existe la carpeta del pipeline: {PIPELINE_DIR}')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

print({'in_colab': IN_COLAB, 'repo_ref': REPO_REF, 'repo_dir': str(REPO_DIR)})

## 2. Configuración protegida

La ejecución completa es pequeña y no necesita repartirse entre cuentas. Revise primero el plan con la bandera en `False`.

In [ ]:
import importlib
import json
import time

import pandas as pd

import ClimateProcessingUtils
import PrecipitationMunicipalAggregation

importlib.reload(ClimateProcessingUtils)
importlib.reload(PrecipitationMunicipalAggregation)

from ClimateProcessingUtils import (
    ahora_proyecto,
    detectar_commit,
    escribir_json_atomico,
    escribir_parquet_atomico,
    escribir_texto_atomico,
    formatear_duracion,
    slugificar,
)
from DatasetConfig import cargar_configuracion_datasets
from PrecipitationMunicipalAggregation import (
    AGGREGATION_VERSION,
    agregar_precipitacion_municipal,
)

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str

    def display(valor):
        print(valor)

if AGGREGATION_VERSION != 'precipitacion_municipio_dia_v1':
    raise RuntimeError(f'Versión municipal inesperada: {AGGREGATION_VERSION}')

VARIABLE_NOMBRE = 'precipitacion'
DATASET_ID = 's54a-sgyg'
CONSOLIDACION_ENTRADA = 'cierre_precipitacion_2024_2025_v2'
CONSOLIDATION_VERSION_ESPERADA = 'precipitacion_estacion_dia_v2'
GEOGRAFIA_ENTRADA = 'estaciones_precipitacion_2024_2025_v3'
GEOGRAPHY_VERSION_ESPERADA = 'climate_station_geography_v3'
AGREGACION_NOMBRE = 'precipitacion_municipio_dia_2024_2025_v1'
FECHA_INICIO = '2024-01-01'
FECHA_FIN = '2025-12-31'
COBERTURA_MINIMA_PCT = 50.0

EJECUTAR_AGREGACION_MUNICIPAL = False
GUARDAR_RESULTADOS = True
SOBRESCRIBIR_RESULTADOS = False

DATASET_CONFIG = cargar_configuracion_datasets(in_colab=IN_COLAB)
PROCESSED_ROOT = DATASET_CONFIG.processed_root
CLIMATE_INPUT_DIR = (
    PROCESSED_ROOT
    / 'clima_diario_curado'
    / f'variable={VARIABLE_NOMBRE}'
    / f'fuente={DATASET_ID}'
    / f'consolidacion={slugificar(CONSOLIDACION_ENTRADA)}'
)
GEOGRAPHY_INPUT_DIR = (
    PROCESSED_ROOT
    / 'geografia_curada'
    / f'canonica={slugificar(GEOGRAFIA_ENTRADA)}'
)
OUTPUT_DIR = (
    PROCESSED_ROOT
    / 'clima_municipal'
    / f'variable={VARIABLE_NOMBRE}'
    / f'fuente={DATASET_ID}'
    / f'agregacion={slugificar(AGREGACION_NOMBRE)}'
)

print({
    'aggregation_version': AGGREGATION_VERSION,
    'ejecutar': EJECUTAR_AGREGACION_MUNICIPAL,
    'guardar': GUARDAR_RESULTADOS,
    'entrada_clima': str(CLIMATE_INPUT_DIR),
    'entrada_geografia': str(GEOGRAPHY_INPUT_DIR),
    'salida': str(OUTPUT_DIR),
})

## 3. Plan y validación de entradas

In [ ]:
def leer_manifest_si_existe(ruta):
    return json.loads(ruta.read_text(encoding='utf-8')) if ruta.exists() else {}


def inspeccionar_plan_municipal():
    clima_manifest = leer_manifest_si_existe(CLIMATE_INPUT_DIR / 'manifest.json')
    geo_manifest = leer_manifest_si_existe(GEOGRAPHY_INPUT_DIR / 'manifest.json')
    particiones = sorted(
        CLIMATE_INPUT_DIR.glob(
            'departamento=*/anio=*/mes=*/observaciones_estacion_dia.parquet'
        )
    )
    return pd.DataFrame([{
        'clima_estado': clima_manifest.get('estado', 'NO_ENCONTRADA'),
        'clima_version': clima_manifest.get('regla_version'),
        'particiones_clima': len(particiones),
        'geografia_estado': geo_manifest.get('estado', 'NO_ENCONTRADA'),
        'geografia_version': geo_manifest.get('geography_version'),
        'estaciones_canonicas': geo_manifest.get('metricas', {}).get('asignaciones_canonicas'),
        'divipola_disponible': (GEOGRAPHY_INPUT_DIR / 'divipola_municipios.parquet').exists(),
        'salida': str(OUTPUT_DIR),
    }])


plan_municipal_df = inspeccionar_plan_municipal()
display(Markdown('### Plan de agregación municipio-día'))
display(plan_municipal_df)

## 4. Funciones de carga, reporte y persistencia

In [ ]:
NOMBRES_SALIDA = {
    'municipio_dia': 'precipitacion_municipio_dia.parquet',
    'resumen': 'resumen_municipios.parquet',
    'grafica': 'cobertura_municipal_diaria.html',
    'reporte': 'AgregacionMunicipal_precipitacion_2024_2025.md',
    'manifest': 'manifest.json',
}


def cargar_entradas_municipales():
    clima_manifest_path = CLIMATE_INPUT_DIR / 'manifest.json'
    geo_manifest_path = GEOGRAPHY_INPUT_DIR / 'manifest.json'
    if not clima_manifest_path.exists() or not geo_manifest_path.exists():
        raise FileNotFoundError('Falta un manifiesto de entrada del paso 05 o 06.')
    clima_manifest = leer_manifest_si_existe(clima_manifest_path)
    geo_manifest = leer_manifest_si_existe(geo_manifest_path)
    if clima_manifest.get('estado') != 'COMPLETA':
        raise RuntimeError('La consolidación estación-día no está COMPLETA.')
    if clima_manifest.get('regla_version') != CONSOLIDATION_VERSION_ESPERADA:
        raise RuntimeError(f'Versión climática inesperada: {clima_manifest.get("regla_version")}')
    if not str(geo_manifest.get('estado', '')).startswith('COMPLETA'):
        raise RuntimeError('La geografía canónica no está completa.')
    if geo_manifest.get('geography_version') != GEOGRAPHY_VERSION_ESPERADA:
        raise RuntimeError(f'Versión geográfica inesperada: {geo_manifest.get("geography_version")}')

    archivos = sorted(
        CLIMATE_INPUT_DIR.glob(
            'departamento=*/anio=*/mes=*/observaciones_estacion_dia.parquet'
        )
    )
    if len(archivos) != 48:
        raise RuntimeError(f'Se esperaban 48 particiones estación-día y existen {len(archivos)}.')
    diario = pd.concat([pd.read_parquet(archivo) for archivo in archivos], ignore_index=True)
    estaciones = pd.read_parquet(GEOGRAPHY_INPUT_DIR / 'estaciones_municipio.parquet')
    divipola = pd.read_parquet(GEOGRAPHY_INPUT_DIR / 'divipola_municipios.parquet')
    return diario, estaciones, divipola, clima_manifest, geo_manifest, archivos


def tabla_markdown(tabla, limite=None):
    vista = tabla.head(limite) if limite is not None else tabla
    try:
        return vista.to_markdown(index=False)
    except ImportError:
        return '```text\n' + vista.to_string(index=False) + '\n```'


def construir_grafica_cobertura(diario_municipal):
    try:
        import plotly.express as px
    except ImportError:
        print('Plotly no está disponible; se omite la gráfica interactiva.')
        return None
    diaria = (
        diario_municipal.groupby('fecha', as_index=False)
        .agg(
            municipios_validos=('es_valido_municipio_dia', 'sum'),
            municipios_con_estacion_esperada=(
                'estaciones_esperadas',
                lambda serie: int(serie.gt(0).sum()),
            ),
            municipios_sin_datos_aceptados=(
                'calidad_municipio_dia',
                lambda serie: int(serie.eq('SIN_DATOS_ACEPTADOS').sum()),
            ),
        )
    )
    larga = diaria.melt(
        id_vars='fecha',
        var_name='serie',
        value_name='municipios',
    )
    figura = px.line(
        larga,
        x='fecha',
        y='municipios',
        color='serie',
        title='Cobertura municipal diaria de precipitación 2024-2025',
        labels={'fecha': 'Fecha', 'municipios': 'Municipios'},
    )
    figura.update_layout(hovermode='x unified', height=520)
    return figura


def construir_reporte_municipal(resultado, clima_manifest, geo_manifest, inicio, fin, duracion):
    calidad = (
        resultado.diario_municipal['calidad_municipio_dia']
        .value_counts()
        .rename_axis('calidad_municipio_dia')
        .reset_index(name='filas')
    )
    return '\n'.join([
        '# Agregación municipal diaria de precipitación',
        '',
        f'- Contrato: `{AGGREGATION_VERSION}`',
        f'- Commit ejecutor: `{detectar_commit(REPO_DIR)}`',
        f'- Commit clima: `{clima_manifest.get("commit")}`',
        f'- Commit geografía: `{geo_manifest.get("commit")}`',
        f'- Inicio: `{inicio.isoformat()}`',
        f'- Fin: `{fin.isoformat()}`',
        f'- Duración: `{formatear_duracion(duracion)}`',
        '',
        '> El valor principal es la mediana no ponderada de estaciones aceptadas. No se imputa y las estadísticas de cobertura permanecen visibles.',
        '',
        '## Métricas',
        '',
        tabla_markdown(pd.DataFrame([resultado.metricas])),
        '',
        '## Calidad municipio-día',
        '',
        tabla_markdown(calidad),
        '',
        '## Municipios con estación canónica',
        '',
        tabla_markdown(
            resultado.resumen_municipio.loc[
                resultado.resumen_municipio['estaciones_canonicas_total'].gt(0)
            ],
        ),
        '',
        '## Decisión',
        '',
        'Este producto es un piloto auditable. Antes de construir indicadores agrícolas se debe revisar cobertura, dispersión multiestación y sensibilidad de la mediana frente a la media.',
        '',
    ])


def guardar_resultado_municipal(resultado, reporte, figura, clima_manifest, geo_manifest, archivos, inicio, fin, duracion):
    manifest_path = OUTPUT_DIR / NOMBRES_SALIDA['manifest']
    if manifest_path.exists() and not SOBRESCRIBIR_RESULTADOS:
        existente = leer_manifest_si_existe(manifest_path)
        if existente.get('estado') == 'COMPLETA':
            print(f'La agregación municipal ya está completa; no se sobrescribe: {OUTPUT_DIR}')
            return existente
    if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()) and not SOBRESCRIBIR_RESULTADOS:
        raise RuntimeError(f'Existe una salida municipal incompleta: {OUTPUT_DIR}')

    escribir_json_atomico({
        'estado': 'INICIADA',
        'aggregation_version': AGGREGATION_VERSION,
        'inicio': inicio.isoformat(),
        'commit': detectar_commit(REPO_DIR),
    }, manifest_path, sobrescribir=SOBRESCRIBIR_RESULTADOS)

    salidas_particiones = []
    diario = resultado.diario_municipal.assign(
        anio=resultado.diario_municipal['fecha'].dt.year,
        mes=resultado.diario_municipal['fecha'].dt.month,
    )
    for (departamento, anio, mes), tabla in diario.groupby(['departamento', 'anio', 'mes'], sort=True):
        ruta = (
            OUTPUT_DIR
            / f'departamento={departamento}'
            / f'anio={int(anio)}'
            / f'mes={int(mes):02d}'
            / NOMBRES_SALIDA['municipio_dia']
        )
        escribir_parquet_atomico(
            tabla.drop(columns=['anio', 'mes']),
            ruta,
            sobrescribir=SOBRESCRIBIR_RESULTADOS,
        )
        salidas_particiones.append({
            'ruta': str(ruta),
            'departamento': departamento,
            'anio': int(anio),
            'mes': int(mes),
            'filas': len(tabla),
            'bytes': ruta.stat().st_size,
        })

    resumen_path = OUTPUT_DIR / NOMBRES_SALIDA['resumen']
    escribir_parquet_atomico(
        resultado.resumen_municipio,
        resumen_path,
        sobrescribir=SOBRESCRIBIR_RESULTADOS,
    )
    reporte_path = OUTPUT_DIR / NOMBRES_SALIDA['reporte']
    escribir_texto_atomico(reporte, reporte_path, sobrescribir=SOBRESCRIBIR_RESULTADOS)
    grafica_path = None
    if figura is not None:
        grafica_path = OUTPUT_DIR / NOMBRES_SALIDA['grafica']
        escribir_texto_atomico(
            figura.to_html(full_html=True, include_plotlyjs='cdn'),
            grafica_path,
            sobrescribir=SOBRESCRIBIR_RESULTADOS,
        )

    manifest = {
        'estado': 'COMPLETA',
        'aggregation_version': AGGREGATION_VERSION,
        'commit': detectar_commit(REPO_DIR),
        'inicio': inicio.isoformat(),
        'fin': fin.isoformat(),
        'duracion_segundos': round(duracion, 2),
        'entrada_clima': {
            'ruta': str(CLIMATE_INPUT_DIR),
            'commit': clima_manifest.get('commit'),
            'regla_version': clima_manifest.get('regla_version'),
            'particiones': len(archivos),
        },
        'entrada_geografia': {
            'ruta': str(GEOGRAPHY_INPUT_DIR),
            'commit': geo_manifest.get('commit'),
            'geography_version': geo_manifest.get('geography_version'),
        },
        'parametros': {
            'fecha_inicio': FECHA_INICIO,
            'fecha_fin': FECHA_FIN,
            'cobertura_minima_pct': COBERTURA_MINIMA_PCT,
            'estadistica_principal': 'MEDIANA_NO_PONDERADA',
        },
        'metricas': resultado.metricas,
        'particiones': salidas_particiones,
        'resumen': {'ruta': str(resumen_path), 'filas': len(resultado.resumen_municipio)},
        'reporte': str(reporte_path),
        'grafica': str(grafica_path) if grafica_path is not None else None,
    }
    escribir_json_atomico(manifest, manifest_path, sobrescribir=True)
    print(f'Agregación municipal guardada en: {OUTPUT_DIR}')
    return manifest

## 5. Ejecución protegida

Cambie únicamente `EJECUTAR_AGREGACION_MUNICIPAL=True` después de confirmar el plan.

In [ ]:
resultado_municipal = None

if not EJECUTAR_AGREGACION_MUNICIPAL:
    print('Agregación municipal desactivada. Revise el plan y active la bandera.')
else:
    inicio = ahora_proyecto()
    reloj = time.perf_counter()
    diario, estaciones, divipola, clima_manifest, geo_manifest, archivos = cargar_entradas_municipales()
    resultado_municipal = agregar_precipitacion_municipal(
        diario,
        estaciones,
        divipola,
        FECHA_INICIO,
        FECHA_FIN,
        cobertura_minima_pct=COBERTURA_MINIMA_PCT,
    )
    controles_cierre = {
        'municipios_objetivo': resultado_municipal.metricas['municipios_objetivo'] == 239,
        'estaciones_canonicas': resultado_municipal.metricas['estaciones_canonicas'] == 116,
        'filas_municipio_dia': resultado_municipal.metricas['filas_municipio_dia'] == 174709,
        'estaciones_no_canonicas_excluidas': resultado_municipal.metricas['estaciones_no_canonicas_excluidas'] == 10,
    }
    if not all(controles_cierre.values()):
        raise RuntimeError(f'Fallaron controles de cierre municipal: {controles_cierre}')
    figura_cobertura = construir_grafica_cobertura(resultado_municipal.diario_municipal)
    fin = ahora_proyecto()
    duracion = time.perf_counter() - reloj
    reporte = construir_reporte_municipal(
        resultado_municipal,
        clima_manifest,
        geo_manifest,
        inicio,
        fin,
        duracion,
    )

    display(Markdown('## Métricas municipales'))
    display(pd.DataFrame([resultado_municipal.metricas]))
    display(Markdown('## Calidad municipio-día'))
    display(
        resultado_municipal.diario_municipal['calidad_municipio_dia']
        .value_counts()
        .rename_axis('calidad_municipio_dia')
        .reset_index(name='filas')
    )
    display(Markdown('## Resumen de municipios con estaciones'))
    display(
        resultado_municipal.resumen_municipio.loc[
            resultado_municipal.resumen_municipio['estaciones_canonicas_total'].gt(0)
        ]
    )
    if figura_cobertura is not None:
        figura_cobertura.show()
    print(f'Duración: {formatear_duracion(duracion)}')

    if GUARDAR_RESULTADOS:
        guardar_resultado_municipal(
            resultado_municipal,
            reporte,
            figura_cobertura,
            clima_manifest,
            geo_manifest,
            archivos,
            inicio,
            fin,
            duracion,
        )
    else:
        print('Resultados no guardados porque GUARDAR_RESULTADOS=False.')

## 6. Compuerta antes de indicadores

La salida todavía es un piloto. Antes del paso 08 se debe:

1. Auditar la distribución de cobertura y los 92 municipio-días preliminares con cobertura insuficiente.
2. Examinar dispersión en municipios multiestación; una diferencia grande puede ser variabilidad espacial real o un problema de estación.
3. Comparar sensibilidad de la mediana frente a la media.
4. Mantener `NaN` para municipios y fechas sin evidencia suficiente.
5. Definir después las ventanas semestrales o de ciclo agrícola.